In [1]:
import pandas as pd
import requests
import io

project_id = "6g4qr"
initial_url = f"https://api.osf.io/v2/nodes/{project_id}/files/osfstorage/"

def load_all_osf_data_with_pagination(start_url):
    all_dfs = []
    current_url = start_url
    page_count = 1

    try:
        print("Starte das Abrufen aller Datensätze von OSF (inkl. Paging)...")
        while current_url:
            print(f"Lade API-Seite {page_count}...")
            response = requests.get(current_url)

            if response.status_code == 200:
                data_json = response.json()
                files_data = data_json.get('data', [])

                for item in files_data:
                    attributes = item.get('attributes', {})
                    name = attributes.get('name', '')
                    kind = attributes.get('kind', '')

                    if kind == 'file' and name.endswith('.csv'):
                        links = item.get('links', {})
                        download_url = links.get('download')

                        if download_url:
                            print(f" -> Lade Datei: {name}")
                            file_response = requests.get(download_url)
                            if file_response.status_code == 200:
                                temp_df = pd.read_csv(io.StringIO(file_response.content.decode('utf-8')))
                                all_dfs.append(temp_df)

                links_pagination = data_json.get('links', {})
                current_url = links_pagination.get('next')
                page_count += 1
            else:
                print(f"Fehler beim API-Abruf (Statuscode: {response.status_code})")
                break

        if all_dfs:
            combined_df = pd.concat(all_dfs, ignore_index=True)
            print(f"\nErfolg! Insgesamt {len(all_dfs)} Probanden-Dateien erfolgreich zusammengefügt.")
            return combined_df
        else:
            print("\nKeine CSV-Dateien gefunden.")
            return pd.DataFrame()

    except Exception as e:
        print(f"Ein Fehler ist aufgetreten: {e}")
        return pd.DataFrame()

# 1. Rohdaten laden
raw_df = load_all_osf_data_with_pagination(initial_url)

# 2. Direktes Preprocessing (Bereinigung)
def preprocess_data(df):
    if df.empty:
        print("Der DataFrame ist leer. Keine Daten zum Bereinigen vorhanden.")
        return df

    # Nur echte Such-Trials behalten (falls die Spalte 'task' existiert)
    if 'task' in df.columns:
        df = df[df['task'] == 'search_trial'].copy()

    # Datentypen konvertieren
    df['rt'] = pd.to_numeric(df['rt'], errors='coerce')
    df['block_num'] = pd.to_numeric(df['block_num'], errors='coerce')
    df['correct'] = df['correct'].astype(bool)

    return df

# Finaler DataFrame 'df' wird erstellt
df = preprocess_data(raw_df)
print(f"Bereinigter Gesamtdatensatz 'df' enthält {len(df)} Zeilen bereit für die Analyse.")

Starte das Abrufen aller Datensätze von OSF (inkl. Paging)...
Lade API-Seite 1...
 -> Lade Datei: data_hul9cyjce0_2026-09-09_12-59.csv
 -> Lade Datei: data_6qdqkbp6a1_2026-09-09_12-58.csv
 -> Lade Datei: data_5d8ue1l50b_2026-09-09_13-2.csv
 -> Lade Datei: data_d0cju5e1cg_2026-09-09_12-55.csv
 -> Lade Datei: data_70r1gcls7n_2026-09-09_12-44.csv
 -> Lade Datei: data_d82x4z6fe8_2026-09-09_12-49.csv
 -> Lade Datei: data_0nkrotxghj_2026-09-09_12-48.csv
 -> Lade Datei: data_tzs2pwwf8f_2026-09-09_12-49.csv
 -> Lade Datei: data_r5ttheuydr_2026-09-09_12-53.csv
 -> Lade Datei: data_lynpy1agod_2026-09-09_12-53.csv
Lade API-Seite 2...
 -> Lade Datei: data_t1p538andz_2026-09-09_12-48.csv
 -> Lade Datei: data_yao8kklm5n_2026-09-09_12-57.csv
 -> Lade Datei: data_vftdbhv4tu_2026-09-09_12-55.csv
 -> Lade Datei: data_aruoysv3x4_2026-09-09_12-44.csv
 -> Lade Datei: data_3ba5upx4pv_2026-09-09_12-44.csv
 -> Lade Datei: data_a69s4z7eer_2026-09-09_12-44.csv

Erfolg! Insgesamt 16 Probanden-Dateien erfolgreich

In [11]:
import numpy as np

def aggregate_data(df):
    df_correct = df[df['correct'] == True].copy()

    # Aggregation auf Block- & Bedingungs-Ebene
    df_block_agg = df_correct.groupby(['block_num', 'condition']).agg(
        mean_rt=('rt', 'mean'),
        std_rt=('rt', 'std'),
        count=('rt', 'count'),
        mean_accuracy=('correct', 'mean')
    ).reset_index()

    df_block_agg['sem_rt'] = df_block_agg['std_rt'] / np.sqrt(df_block_agg['count'])

    # Aggregation auf Personen-Ebene
    df_subject_agg = df_correct.groupby(['subject', 'condition']).agg(
        mean_rt=('rt', 'mean'),
        mean_accuracy=('correct', 'mean')
    ).reset_index()

    return df_block_agg, df_subject_agg

In [12]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

sns.set_theme(style="whitegrid")

def plot_contextual_cueing(df_block_agg):
    plt.figure(figsize=(10, 6))

    # Block-Nummer für die Regression in numerisch wandeln
    df_block_agg['block_num_int'] = df_block_agg['block_num'].astype(int)

    # 1. Dezente Trendlinien (Regression) im Hintergrund
    sns.regplot(
        data=df_block_agg[df_block_agg['condition'] == 'old'],
        x='block_num_int',
        y='mean_rt',
        scatter=False,
        color='#1f77b4',
        line_kws={'linestyle': '--', 'alpha': 0.4, 'linewidth': 1.5}
    )
    sns.regplot(
        data=df_block_agg[df_block_agg['condition'] == 'new'],
        x='block_num_int',
        y='mean_rt',
        scatter=False,
        color='#ff7f0e',
        line_kws={'linestyle': '--', 'alpha': 0.4, 'linewidth': 1.5}
    )

    # 2. Eigentliche Haupt-Lernkurve (Mittelwerte mit Punkten) im Vordergrund
    ax = sns.lineplot(
        data=df_block_agg,
        x='block_num_int',
        y='mean_rt',
        hue='condition',
        marker='o',
        linewidth=2.5,
        markersize=7,
        palette={'old': '#1f77b4', 'new': '#ff7f0e'}
    )

    plt.title('Contextual Cueing Effekt: Lernkurve mit dezenten Trendlinien', fontsize=14, fontweight='bold')
    plt.xlabel('Experimenteller Block', fontsize=12)
    plt.ylabel('Mittlere Reaktionszeit (RT in ms)', fontsize=12)

    # Saubere Legende nur für die Hauptlinien
    handles, labels = ax.get_legend_handles_labels()
    plt.legend(handles=handles, labels=['Wiederholt (Old)', 'Zufällig (New)'], title='Bedingung', loc='upper right')

    plt.tight_layout()
    plt.show()

In [4]:
from scipy import stats

def check_assumptions(df_subject_agg):
    print("--- 1. Shapiro-Wilk-Test auf Normalverteilung ---")
    for condition in df_subject_agg['condition'].unique():
        subset = df_subject_agg[df_subject_agg['condition'] == condition]['mean_rt']
        stat, p_value = stats.shapiro(subset)
        print(f"Bedingung '{condition}': p-Wert = {p_value:.4f}")

    print("\n--- 2. Levene-Test auf Varianzhomogenität ---")
    old_rt = df_subject_agg[df_subject_agg['condition'] == 'old']['mean_rt']
    new_rt = df_subject_agg[df_subject_agg['condition'] == 'new']['mean_rt']
    levene_stat, levene_p = stats.levene(old_rt, new_rt)
    print(f"Levene-Test p-Wert = {levene_p:.4f}")

In [5]:
def run_inferential_stats(df_subject_agg):
    old_data = df_subject_agg[df_subject_agg['condition'] == 'old'].sort_values('subject')['mean_rt'].values
    new_data = df_subject_agg[df_subject_agg['condition'] == 'new'].sort_values('subject')['mean_rt'].values

    t_stat, p_val = stats.ttest_rel(old_data, new_data)

    diff = old_data - new_data
    cohens_d = np.mean(diff) / np.std(diff, ddof=1)

    print(f"t-Statistik: {t_stat:.4f} | p-Wert: {p_val:.5f} | Cohen's d: {cohens_d:.4f}")

In [6]:
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns

def run_multilevel_model(df):
    df_correct = df[df['correct'] == True].copy()

    # MLM fitten
    model = smf.mixedlm("rt ~ condition", df_correct, groups=df_correct["subject"])
    result = model.fit()
    print(result.summary())

    plt.figure(figsize=(10, 6))

    df_correct['block_num_int'] = df_correct['block_num'].astype(int)
    df_indiv = df_correct.groupby(['block_num_int', 'subject', 'condition'])['rt'].mean().reset_index()

    # 1. Individuelle Probanden-Linien im Hintergrund (sehr dezent)
    sns.lineplot(
        data=df_indiv,
        x='block_num_int',
        y='rt',
        units='subject',
        estimator=None,
        color='gray',
        alpha=0.2,
        linewidth=0.8
    )

    # 2. Dezente Regressions-Trendlinien für Old und New über alle Daten hinweg
    sns.regplot(
        data=df_indiv[df_indiv['condition'] == 'old'],
        x='block_num_int',
        y='rt',
        scatter=False,
        color='blue',
        line_kws={'linestyle': '-', 'alpha': 0.35, 'linewidth': 2}
    )
    sns.regplot(
        data=df_indiv[df_indiv['condition'] == 'new'],
        x='block_num_int',
        y='rt',
        scatter=False,
        color='orange',
        line_kws={'linestyle': '--', 'alpha': 0.35, 'linewidth': 2}
    )

    # 3. Aggregierte Haupt-Mittelwertslinien darüber legen
    sns.lineplot(
        data=df_indiv,
        x='block_num_int',
        y='rt',
        hue='condition',
        marker='s',
        linewidth=2.5,
        palette={'old': 'blue', 'new': 'orange'}
    )

    plt.title('Multilevel-Modell: Probanden-Verläufe und dezente Trends', fontsize=14, fontweight='bold')
    plt.xlabel('Experimenteller Block', fontsize=12)
    plt.ylabel('Reaktionszeit (RT in ms)', fontsize=12)

    # Legende manuell sauber setzen
    handles, labels = plt.gca().get_legend_handles_labels()
    # Wir nehmen nur die Hauptelemente für die Legende (ignorieren die Regplot-Einträge)
    plt.legend(handles=handles[-2:], labels=['Alt (Old)', 'Neu (New)'], loc='upper right', title='Bedingung')

    plt.tight_layout()
    plt.show()

In [8]:
# ==========================================
# AUSFÜHRUNG DER ANALYSEN
# ==========================================

print(anzahl_zeilen := f"Anzahl Zeilen im DataFrame: {len(df)}")

if not df.empty:
    # 1. Task 2: Daten aggregieren
    print("\n--- Führe Task 2 aus: Aggregation ---")
    df_block_agg, df_subject_agg = aggregate_data(df)
    print("Aggregation erfolgreich abgeschlossen.")

    # 2. Task 3: Visualisierung (Lernkurve)
    print("\n--- Führe Task 3 aus: Visualisierung ---")
    plot_contextual_cueing(df_block_agg)

    # 3. Task 4: Voraussetzungen prüfen
    print("\n--- Führe Task 4 aus: Assumption Checks ---")
    check_assumptions(df_subject_agg)

    # 4. Task 5: Inferenzstatistik (t-Test & Effektstärke)
    print("\n--- Führe Task 5 aus: Inferenzstatistik ---")
    run_inferential_stats(df_subject_agg)

    # 5. Task 6: Multilevel Model & Spaghetti-Plot
    print("\n--- Führe Task 6 aus: Multilevel Modeling ---")
    run_multilevel_model(df)
else:
    print("Achtung: Der DataFrame 'df' ist leer. Bitte prüfe, ob deine CSV-Dateien auf OSF Daten enthalten und ob das Projekt öffentlich ist.")

NameError: name 'df' is not defined